In [ ]:
# Import Libraries
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse
import os

Configuration

In [ ]:
print(os.getcwd())

In [ ]:
file_path = "./rna_malignant_counts.h5ad"

print(f"Loading {file_path}...")
adata = sc.read_h5ad(file_path)
print(f"Initial Shape: {adata.shape}")

In [ ]:
adata

1. METADATA PREP (Condition & Cell Type)

In [ ]:
#FIRST # A. Create 'condition' column
# # Critical: 'ctrl' must be explicitly defined
# condition_map = {
#     'pre-IR': 'ctrl',
#     '24h post-IR': 'IR_24h',
#     '1 week post-IR': 'IR_1week'
# }
#
# # Check if Timepoint exists
# if 'Timepoint' not in adata.obs:
#     raise KeyError("CRITICAL: 'Timepoint' column missing in raw data!")
#
# adata.obs['condition'] = adata.obs['Timepoint'].map(condition_map)
# adata.obs['condition'] = adata.obs['condition'].astype('category')
# #
# # B. Create 'cell_type' column (Cleaned string)
# if 'GBM_malig_subtypes' in adata.obs:
#     adata.obs['cell_type'] = adata.obs['GBM_malig_subtypes'].str.replace("-", "_")
#     adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')
#
# print(f"Conditions found: {adata.obs['condition'].unique().tolist()}")

In [ ]:
# ==========================================
# 1. CREATE CONDITION / CELL TYPE METADATA
# ==========================================

# A. Create 'condition' column
# ctrl = pre-radiotherapy baseline
# IR_24h and IR_1week are metadata labels, NOT genes
condition_map = {
    "pre-IR": "ctrl",
    "24h post-IR": "IR_24h",
    "1 week post-IR": "IR_1week",
}

if "Timepoint" not in adata.obs:
    raise KeyError("CRITICAL: 'Timepoint' column missing in raw data!")

adata.obs["condition"] = adata.obs["Timepoint"].map(condition_map)

# Check for unmapped timepoints
if adata.obs["condition"].isna().any():
    print("Unmapped Timepoint values:")
    print(adata.obs.loc[adata.obs["condition"].isna(), "Timepoint"].unique())
    raise ValueError("Some Timepoint values were not mapped to condition labels.")

adata.obs["condition"] = adata.obs["condition"].astype("category")

# B. Duplicate condition into condition_name
# Some scGPT/GEARS utilities expect this column
adata.obs["condition_name"] = adata.obs["condition"].astype(str).astype("category")

# C. Create 'cell_type' column
if "GBM_malig_subtypes" not in adata.obs:
    raise KeyError("CRITICAL: 'GBM_malig_subtypes' column missing in raw data!")

adata.obs["cell_type"] = (
    adata.obs["GBM_malig_subtypes"]
    .astype(str)
    .str.replace("-", "_", regex=False)
    .astype("category")
)

print("Conditions found:")
print(adata.obs["condition"].value_counts())

print("\nCell types found:")
print(adata.obs["cell_type"].value_counts())

2. GENE NAME PREP (Mouse -> Human)

In [ ]:
# import pickle
#
# # ==========================================
# # 2. GENE MAPPING & SANITIZATION
# # ==========================================
#
# # Loading the mapping file
# orthology_file = "./mouse2human.pkl"
# print("\n[Step 2/4] Mapping Mouse Genes to Human Orthologs...")
#
#
# # Load mapping
# try:
#     with open(orthology_file, "rb") as f:
#         mouse2human = pickle.load(f)
#     print(f"Loaded mapping for {len(mouse2human)} genes.")
# except FileNotFoundError:
#     raise FileNotFoundError(f"Missing {orthology_file}. Please ensure it is in the folder.")
#
# # A. Filter: Keep only mouse genes that have a Human partner
# valid_mouse_genes = [g for g in adata.var_names if g in mouse2human]
# print(f"Filtering: {len(adata.var_names)} original -> {len(valid_mouse_genes)} mapped genes.")
#
# adata = adata[:, valid_mouse_genes].copy()
#
# # B. Rename to Human Symbols
# new_human_names = [mouse2human[g] for g in adata.var_names]
# adata.var.index = pd.Index(new_human_names)
#
# if not adata.var_names.is_unique:
#     print("Resolving duplicate human gene names...")
#     adata.var_names_make_unique()
#
# # Save 'gene_name' column (Required by loader)
# adata.var['gene_name'] = adata.var.index
#
# # --- CRITICAL FIX FOR INFINITY ERROR ---
# print("Sanitizing expression matrix...")
#
# # 1. Store Raw Counts explicitly (Enables 'seurat_v3' flavor)
# # If 'counts' layer is missing, we assume X holds the counts
# if "counts" not in adata.layers:
#     adata.layers["counts"] = adata.X.copy()
#
# # 2. Check for Infinity/NaNs and fix them
# if np.isinf(adata.X.data).any() if sparse.issparse(adata.X) else np.isinf(adata.X).any():
#     print("⚠️ Found Infinity values in X! Replacing with max finite value.")
#     # Replace inf with a large number (or 0 if preferred, but max finite preserves rank)
#     # For sparse matrices, this is tricky, simpler to zero them out or clip.
#     if sparse.issparse(adata.X):
#         adata.X.data = np.nan_to_num(adata.X.data, posinf=1e5, neginf=0)
#     else:
#         adata.X = np.nan_to_num(adata.X, posinf=1e5, neginf=0)
#
# # 3. Remove genes expressed in 0 cells (prevents divide-by-zero errors)
# sc.pp.filter_genes(adata, min_cells=1)
#
# # C. Filter to Top 3000 HVGs
# print("Filtering to Top 3000 Highly Variable Genes...")
#
# # We force 'seurat_v3' because we now GUARANTEE the 'counts' layer exists above.
# # seurat_v3 is much more robust for count data than the default 'seurat'.
# sc.pp.highly_variable_genes(
#     adata,
#     n_top_genes=3000,
#     layer="counts",
#     flavor="seurat_v3",
#     subset=True,
#     span=0.8 # fixes error with SVD in seurat_v3 (LOESS curve fitting)
# )

In [ ]:
# # ==========================================================
# # STEP 2.5: NORMALIZE DATA PERMANENTLY
# # ==========================================================
# print("Applying Log-Normalization...")
#
# # 1. Normalize library size (standardize read depth)
# sc.pp.normalize_total(adata, target_sum=1e4)
#
# # 2. Logarithmize the data
# #    This fixes "AssertionError: gene expression should be log transformed"
# sc.pp.log1p(adata)
#
# print("Normalization complete. Max value:", adata.X.max())

In [ ]:
import pickle

# ==========================================
# 2. GENE MAPPING, NORMALIZATION & FILTERING
# ==========================================

orthology_file = "./mouse2human.pkl"
print("\n[Step 2/4] Mapping Mouse Genes to Human Orthologs...")

# --- A. Load Mapping ---
try:
    with open(orthology_file, "rb") as f:
        mouse2human = pickle.load(f)
    print(f"Loaded mapping for {len(mouse2human)} genes.")
except FileNotFoundError:
    raise FileNotFoundError(f"Missing {orthology_file}. Please ensure it is in the folder.")

# --- B. Filter & Rename ---
valid_mouse_genes = [g for g in adata.var_names if g in mouse2human]
print(f"Filtering: {len(adata.var_names)} original -> {len(valid_mouse_genes)} mapped genes.")

adata = adata[:, valid_mouse_genes].copy()

new_human_names = [mouse2human[g] for g in adata.var_names]
adata.var.index = pd.Index(new_human_names)

# if not adata.var_names.is_unique:
#     print("Resolving duplicate human gene names...")
#     adata.var_names_make_unique()

adata.var['gene_name'] = adata.var.index

# --- C. Sanitize & Filter ---
print("Sanitizing expression matrix...")

# 1. Store Raw Counts (Critical for record keeping)
if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()

# 2. Fix Infinity/NaN
if np.isinf(adata.X.data).any() if sparse.issparse(adata.X) else np.isinf(adata.X).any():
    print("⚠️ Found Infinity values in X! Replacing with max finite value.")
    if sparse.issparse(adata.X):
        adata.X.data = np.nan_to_num(adata.X.data, posinf=1e5, neginf=0)
    else:
        adata.X = np.nan_to_num(adata.X, posinf=1e5, neginf=0)

# 3. Filter genes expressed in very few cells (Increase to 3 for stability)
sc.pp.filter_genes(adata, min_cells=3)

# --- D. NORMALIZE (Moved here to support standard HVG) ---
# We perform the normalization NOW so we can use the stable 'seurat' flavor
print("Applying Log-Normalization...")
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print(f"Normalization complete. Max value: {adata.X.max():.2f}")

# # --- E. HVG Selection (Stable Method) ---
# print("Filtering to Top 3000 Highly Variable Genes...")
#
# # We use flavor='seurat' (the default) instead of 'seurat_v3'.
# # This works on the log-normalized data we just created and avoids the "Chernobyl" crash.
# sc.pp.highly_variable_genes(
#     adata,
#     n_top_genes=None,
#     flavor="seurat",  # <--- CHANGED: More stable method
#     subset=True
# )

# --- E. HVG Selection (Using Pre-computed Author Labels) ---
print("Filtering using the pre-computed 'highly_variable' column...")

# Simply slice the AnnData object to only keep genes where this column is True
adata = adata[:, adata.var['highly_variable'] == True].copy()

print(f"Final data shape after applying author HVGs: {adata.shape}")

# print(f"Final data shape: {adata.shape}")
# ==========================================
# VALIDATE: TIMEPOINT LABELS ARE NOT GENES
# ==========================================

adata.var["gene_name"] = adata.var.index.astype(str)

timepoint_tokens = {"IR_24h", "IR_1week"}
gene_set = set(adata.var["gene_name"])

overlap = timepoint_tokens.intersection(gene_set)

if overlap:
    raise ValueError(
        f"Timepoint labels should not be in adata.var['gene_name']: {overlap}"
    )
else:
    print("✅ Timepoint labels are metadata only, not genes.")

3. DUMMY GENES (For Environmental Perturbations)

In [ ]:
# print("\n[Step 3/4] Injecting Dummy Genes...")
#
# conditions = adata.obs['condition'].unique()
# existing_genes = set(adata.var['gene_name'])
# missing_perts = list(set([p for c in conditions if c != 'ctrl' for p in c.split('+') if p not in existing_genes]))
#
# if missing_perts:
#     print(f"Adding dummy genes: {missing_perts}")
#
#     new_X = sparse.csr_matrix((adata.shape[0], len(missing_perts)), dtype=np.float32)
#     var_df = pd.DataFrame(index=missing_perts)
#     var_df['gene_name'] = missing_perts
#
#     # --- SMART FILLING TO PREVENT TYPE ERRORS ---
#     for col in adata.var.columns:
#         if col == 'gene_name': continue
#
#         # Check the type of the original column
#         if pd.api.types.is_bool_dtype(adata.var[col]):
#             var_df[col] = False  # Boolean columns get False
#         elif pd.api.types.is_numeric_dtype(adata.var[col]):
#             var_df[col] = 0.0    # Numeric columns get 0
#         else:
#             var_df[col] = "dummy" # String columns get "dummy"
#
#     adata_dummy = sc.AnnData(X=new_X, obs=adata.obs, var=var_df)
#     adata = sc.concat([adata, adata_dummy], axis=1)
#
#     # Restore metadata
#     adata.obs = adata_dummy.obs.copy()
#     adata.var['gene_name'] = adata.var.index
# else:
#     print("No dummy genes needed.")

In [ ]:
# import scipy.sparse as sparse # Ensure this is imported
#
# print("\n[Step 3/4] Injecting Dummy Genes...")
#
# conditions = adata.obs['condition'].unique()
# existing_genes = set(adata.var['gene_name'])
# # This list comprehension handles single conditions (IR_24h) AND combinitorials (A+B)
# missing_perts = list(set([p for c in conditions if c != 'ctrl' for p in c.split('+') if p not in existing_genes]))
#
# if missing_perts:
#     print(f"Adding dummy genes: {missing_perts}")
#
#     # ======================================================
#     # NEW LOGIC: Create Binary Flags (1.0 or 0.0)
#     # ======================================================
#     dummy_cols = []
#
#     for pert in missing_perts:
#         # Check each cell: Does its condition string contain this perturbation?
#         # We use .apply() to handle cases like "PertA+PertB" safely
#         # .astype(np.float32) converts True->1.0 and False->0.0
#         binary_flag = adata.obs['condition'].apply(lambda x: pert in x.split('+')).astype(np.float32).values
#
#         # Reshape to a column vector (N_cells x 1) and make sparse
#         col_sparse = sparse.csr_matrix(binary_flag.reshape(-1, 1))
#         dummy_cols.append(col_sparse)
#
#     # Stack all columns horizontally to create the final matrix (N_cells x N_dummy_genes)
#     new_X = sparse.hstack(dummy_cols)
#     # ======================================================
#
#     var_df = pd.DataFrame(index=missing_perts)
#     var_df['gene_name'] = missing_perts
#
#     # --- SMART FILLING TO PREVENT TYPE ERRORS ---
#     for col in adata.var.columns:
#         if col == 'gene_name': continue
#
#         if pd.api.types.is_bool_dtype(adata.var[col]):
#             var_df[col] = False
#         elif pd.api.types.is_numeric_dtype(adata.var[col]):
#             var_df[col] = 0.0
#         else:
#             var_df[col] = "dummy"
#
#     adata_dummy = sc.AnnData(X=new_X, obs=adata.obs, var=var_df)
#     adata = sc.concat([adata, adata_dummy], axis=1)
#
#     # Restore metadata
#     adata.obs = adata_dummy.obs.copy()
#     adata.var['gene_name'] = adata.var.index
# else:
#     print("No dummy genes needed.")

In [ ]:
import scipy.sparse as sparse

############# print("\n[Step 3/4] Injecting Dummy Genes...")

conditions = adata.obs['condition'].unique()
existing_genes = set(adata.var['gene_name'])
missing_perts = list(set([p for c in conditions if c != 'ctrl' for p in c.split('+') if p not in existing_genes]))

if missing_perts:
    print(f"Adding dummy genes: {missing_perts}")

    # --- NEW LOGIC: Create Binary Flags (1.0) ---
    dummy_cols = []
    for pert in missing_perts:
        # Create a 1.0/0.0 vector for this perturbation
        binary_flag = adata.obs['condition'].apply(lambda x: pert in x.split('+')).astype(np.float32).values
        dummy_cols.append(sparse.csr_matrix(binary_flag.reshape(-1, 1)))

    new_X = sparse.hstack(dummy_cols)
    # --------------------------------------------

    var_df = pd.DataFrame(index=missing_perts)
    var_df['gene_name'] = missing_perts

    # Smart filling for metadata
    for col in adata.var.columns:
        if col == 'gene_name': continue
        if pd.api.types.is_bool_dtype(adata.var[col]):
            var_df[col] = False
        elif pd.api.types.is_numeric_dtype(adata.var[col]):
            var_df[col] = 0.0
        else:
            var_df[col] = "dummy"

    adata_dummy = sc.AnnData(X=new_X, obs=adata.obs, var=var_df)
    adata = sc.concat([adata, adata_dummy], axis=1)

    # Restore metadata
    adata.obs = adata_dummy.obs.copy()
    adata.var['gene_name'] = adata.var.index
else:
    print("No dummy genes needed.")

4. DE STATS (Required for Graph Construction)

In [ ]:
print("\n[Step 4/4] Calculating DE Stats...")

# 1. Create a working copy
# CRITICAL: 'adata' is ALREADY normalized from Step 2.5.
# Do NOT run normalize_total or log1p here again.
adata_norm = adata.copy()

try:
    # 2. Run Rank Genes Groups (Wilcoxon)
    # We use 'use_raw=False' to ensure it uses the normalized .X we just prepared
    sc.tl.rank_genes_groups(
        adata_norm,
        groupby='condition',
        reference='ctrl',
        method='wilcoxon',
        use_raw=False
    )

    # 3. Save the results back to the main object
    # GEARS specifically looks for 'rank_genes_groups_cov_all'
    adata.uns['rank_genes_groups_cov_all'] = adata_norm.uns['rank_genes_groups']
    print("DE Stats calculated successfully.")

except Exception as e:
    print(f"ERROR in DE calculation: {e}")

New Emerged Problem: column_name\
Solution: Duplicate the "condition" to a new column "condition_name"

In [ ]:
import scanpy as sc

# 1. Load the file
# file_path = "./data/project_dataset/perturb_processed.h5ad"
# print(f"Loading {file_path}...")
# adata = sc.read_h5ad(file_path)

# 2. Duplicate 'condition' into 'condition_name'
# This satisfies the library's redundant requirement
print("Creating 'condition_name' column...")
adata.obs['condition_name'] = adata.obs['condition']

# 3. Verify it exists
print(f"Columns in obs: {adata.obs.columns.tolist()}")
print(f"Sample values: {adata.obs[['condition', 'condition_name']].head(3)}")
#
# # 4. Save
# print(f"Saving patched file to {file_path}...")
# adata.write_h5ad(file_path, compression="gzip")
# print("✅ Success! You can now run pert_data.load() again.")


New Emerged Problem: Error Key IR_1week; GEARS is expecting a simple python dictionary i.e., {'IR_1week': ['GeneA', ...] where scanpy outputs as array i.e., {'names': array(...), 'scores': array(...)\
Solution: Extract gene names from Scanpy stats and reformats them into simple python dictionary for GEARS to operate.

In [ ]:
# 2. Check if the raw Scanpy stats exist
if 'rank_genes_groups_cov_all' not in adata.uns:
    raise KeyError("Could not find 'rank_genes_groups_cov_all'. Please run the previous DE step first.")

raw_de = adata.uns['rank_genes_groups_cov_all']
print("Found raw Scanpy DE stats. Converting to GEARS dictionary format...")

new_de_dict = {}

# --- UNIVERSAL FIX ---
# 1. Force conversion to DataFrame.
#    This works whether 'names' is a structured array OR already a DataFrame.
names_df = pd.DataFrame(raw_de['names'])

# 2. Iterate through columns (conditions)
for group in names_df.columns:
    # Get the column as a list
    gene_list = names_df[group].tolist()

    # 3. Byte string decoding (Safety check)
    if gene_list and isinstance(gene_list[0], bytes):
        gene_list = [g.decode('utf-8') for g in gene_list]

    new_de_dict[group] = gene_list
    print(f"  > Processed group '{group}': {len(gene_list)} genes")

# 4. Save back
adata.uns['rank_genes_groups_cov_all'] = new_de_dict
print("Conversion complete.")

In [ ]:
adata

In [ ]:
# # ... [Your existing code that creates the 'adata' object] ...
#
# # ==========================================================
# # NEW: NORMALIZE DATA PERMANENTLY
# # ==========================================================
# print("Applying Log-Normalization...")
#
# # 1. Normalize library size (standardize read depth)
# #    This makes sure a cell with 10,000 reads is comparable to one with 5,000.
# sc.pp.normalize_total(adata, target_sum=1e4)
#
# # 2. Logarithmize the data
# #    This squashes giant counts (e.g., 5000) down to small floats (e.g., ~8.5)
# #    This fixes the "AssertionError: gene expression should be log transformed"
# sc.pp.log1p(adata)
#
# print("Normalization complete. Max value in data:", adata.X.max())
# # ==========================================================


Exporting

In [ ]:
# The output location required by scGPT/GEARS
# output_dir = "./"
output_dir = "./data/adamson"
output_file = os.path.join(output_dir, "perturb_processed.h5ad")

os.makedirs(output_dir, exist_ok=True)

print(f"\nSaving to: {output_file}")
adata.write_h5ad(output_file, compression="gzip")

In [ ]:
# print("=== AnnData Info ===")
# print(f"Shape: {adata.shape}")
# print(f"Obs columns: {adata.obs.columns.tolist()}")
# print(f"Var columns: {adata.var.columns.tolist()}")
#
# # Check gene names
# gene_names = adata.var['gene_name'].values if 'gene_name' in adata.var else adata.var_names
# print(f"\nFirst 10 gene names: {gene_names[:10]}")
#
# # Look for dummy genes
# dummy_candidates = ['IR-24h', 'IR-1week', 'IR_24h', 'IR_1week']
# found_dummies = [g for g in dummy_candidates if g in gene_names]
# if found_dummies:
#     print(f"\n⚠️ Dummy genes found: {found_dummies}")
# else:
#     print("\n✅ No dummy genes found. Good!")
#
# # Check condition labels
# print(f"\nUnique conditions: {adata.obs['condition'].unique()}")

In [ ]:
# # In your notebook before calling pert_data.load(...)
# import numpy as np
# import anndata as ad
#
# adata = ad.read_h5ad("./data/adamson0/perturb_processed.h5ad")
# gene_names_set = set(adata.var.gene_name)
# print("Condition values:")
# for cond in adata.obs['condition'].unique():
#     for g in cond.split('+'):
#         if g != 'ctrl' and g not in gene_names_set:
#             print(f"Gene '{g}' not found in var.gene_name!")

In [ ]:
# import anndata as ad
# import shutil
#
# # Backup original
# shutil.copy("./data/adamson0/perturb_processed.h5ad",
#             "./data/adamson0/perturb_processed_backup.h5ad")
#
# # Load
# adata = ad.read_h5ad("./data/adamson0/perturb_processed.h5ad")
#
# # Remove cells where condition is exactly one of the dummy labels
# bad_conditions = ['IR_1week', 'IR_24h']
# mask = ~adata.obs['condition'].isin(bad_conditions)
# adata = adata[mask].copy()
#
# # Save the cleaned file (overwrites the original)
# adata.write_h5ad("./data/adamson0/perturb_processed.h5ad")

In [ ]:
adata.obs

Including dummy indices to the vocab.json

In [ ]:
# import json
# import os
#
# # CONFIGURATION
# vocab_path = "save/scGPT_human/vocab.json"  # Path to your existing vocab file
# new_tokens = ["IR_24h", "IR_1week"]         # The tokens to add
#
# def update_vocab(file_path, new_tokens):
#     # 1. Load the existing vocabulary
#     if not os.path.exists(file_path):
#         print(f"Error: File not found at {file_path}")
#         return
#
#     with open(file_path, 'r') as f:
#         vocab = json.load(f)
#
#     print(f"Original vocab size: {len(vocab)}")
#
#     # 2. Check if tokens already exist to avoid duplicates
#     existing_keys = set(vocab.keys())
#     tokens_to_add = [t for t in new_tokens if t not in existing_keys]
#
#     if not tokens_to_add:
#         print("All tokens already exist in vocabulary. No changes made.")
#         return
#
#     # 3. Find the current highest ID
#     # We take the max of all values to ensure we append strictly to the end
#     current_ids = vocab.values()
#     next_id = max(current_ids) + 1
#
#     # 4. Append new tokens
#     for token in tokens_to_add:
#         vocab[token] = next_id
#         print(f"Adding token: '{token}' -> ID: {next_id}")
#         next_id += 1
#
#     # 5. Save the updated vocabulary
#     # We use indent=2 to keep it readable, or None for compact size
#     with open(file_path, 'w') as f:
#         json.dump(vocab, f, indent=2)
#
#     print(f"Success! Updated vocab size: {len(vocab)}")
#
# if __name__ == "__main__":
#     update_vocab(vocab_path, new_tokens)

In [ ]:
# import scanpy as sc
# import os
#
# h5ad_path = "./data/radiotherapyGB/perturb_processed.h5ad"
#
# print("File exists:", os.path.exists(h5ad_path))
# adata_check = sc.read_h5ad(h5ad_path)
#
# print("Shape:", adata_check.shape)
# print("obs columns:")
# print(adata_check.obs.columns.tolist())
#
# print("\nvar columns:")
# print(adata_check.var.columns.tolist())